<h1 style="color:#1F4E78;">Flujo de efectivo</h1>

In [7]:
import pandas as pd
from openpyxl.styles import Font, Border, Side, Alignment, PatternFill
from openpyxl.utils import get_column_letter

df = pd.read_excel("Datos flujo de efectivo.xlsx")

df["inicio_semana"] = pd.to_datetime(df["inicio_semana"], dayfirst=True)

df["concepto_agrupado"] = (df["descripción"].astype(str).str.strip().str.upper())

equivalencias_conceptos = {

    # Proberry
    "PROBERRY": "PROBERRY",
    "PROBERRY 80/20": "PROBERRY",
    "PROBERRY 80/20 JULIO CAJA 10 LBS": "PROBERRY",
    "PROBERRY 80/20 JULIO CAJA 20 LBS": "PROBERRY",
    "PROBERRY ARANDANO": "PROBERRY",
    "PROBERRY BLOQUE FRAM": "PROBERRY",
    "PROBERRY CRUMBLE": "PROBERRY",
    "PROBERRY CRUMBLE AGUASCALIENTES": "PROBERRY",
    "PROBERRY CRUMBLE MARITIMO 1/2": "PROBERRY",
    "CUBETA CRUMBLE AGUSCALIENTES": "PROBERRY",
    "CRUMBLE LUIS GLEZ": "PROBERRY",
    "PROBERRY ZARZA": "PROBERRY",
    "PROBERRY ZARZAMORA": "PROBERRY",
    "PROBERRY A CUENTA": "PROBERRY",
    "PROBERRY A CUENTA ( PAGO POR CONTADOR EFECTIVO)": "PROBERRY",
    "PROBERRY BARRILES": "PROBERRY",

    # Nómina
    "NOMINA CXM": "NOMINA CXM",
    "NOMINA JACONA SABADO": "NOMINA JACONA",
    "NOMINA JACONA MIERCOLES": "NOMINA JACONA",
    "NOMINA JACONA Y OCUMICHO SABADO": "NOMINA JACONA",
    "OMAR": "OMAR GUERRERO",
    "OMAR NOMINA": "OMAR GUERRERO",
    "ALEJANDRO RODRIGUEZ": "ALEJANDRO RODRIGUEZ",

    # AVV
    "AVV  (MONARCA)": "AVV MONARCA",
    "AVV (PAGO MONARCA)": "AVV MONARCA",
    "AVV (PAGO SANTANDER)": "AVV PAGO SANTANDER",
    "AVV (PAGO SANTANDERT)": "AVV PAGO SANTANDER",

    # Materiales
    "CAJA CARTON (PATAGONIA)": "PATAGONIA CAJA CARTON",
    "CAJA CARTON (PATAGONIA) 20 Y 30 LBS": "PATAGONIA CAJA CARTON",
    "CAJA PATAGONIA": "PATAGONIA CAJA CARTON",
    "PATAGONIA CAJA CARTON Y 30 LBS": "PATAGONIA CAJA CARTON",
    "CAJA RUTH GDLMK BOXES": "CAJA RUTH GDL",

    "EMBAPACK BOLSA AZUL": "EMBAPACK",
    "EMBAPACK BOLSA AZUL Y VERDE": "EMBAPACK",
    "EMBAPACK BOLSA AZUL)": "EMBAPACK",
    "EMBAPACK BOLSA TRNASPARENTE": "EMBAPACK",
    "EMBAPACK BOLSA VERDE": "EMBAPACK",
    "EMBAPACK BOLSA VERDE)": "EMBAPACK",

    "CODICE": "CODICE ETIQUETAS",
    "ETIQUETAS": "CODICE ETIQUETAS",
    "ETIQUETAS CODICE": "CODICE ETIQUETAS",

    # Impuestos / IMSS
    "IMPUESTOS LAR FEBRERO": "IMPUESTOS LAR",
    "FACTURA E IMPUESTOS LAR": "IMPUESTOS LAR",
    "IMSS CXM": "IMSS",
    "IMSS FV": "IMSS",
    "IMSS (DEPOSITO A CUENTA AVV)": "IMSS",
    "CXM": "IMSS",

    # Otros repetidos
    "PRIMUS LAB": "PRIMUS LAB",
    "CHURRAS": "CHURRAS",
    "GUS": "GUSTAVO",
    "GUS GASTOS": "GUSTAVO GASTOS",
    "GUSTAVO GASTOS": "GUSTAVO GASTOS",
    "VERO": "VERO",
    "FRANCISCO (CANCUN)": "FRANCISCO CANCUN",
    "GARNICA CONSULTING (FACTURA 415)": "GARNICA CONSULTING",
    "GARNICA CONSULTING (FACTURA 436)": "GARNICA CONSULTING",
    "MARACUYA BORREGOS": "MARACUYA",
    "SALDO LAR": "LAR",
}

df["concepto_agrupado"] = df["concepto_agrupado"].replace(equivalencias_conceptos)

df["importe_flujo"] = df["importe"]
df.loc[df["tipo"] == "Egreso", "importe_flujo"] *= -1

flujo = (df.groupby(["concepto_agrupado", "inicio_semana"])["importe_flujo"].sum().unstack(fill_value=0))
flujo = flujo.sort_index()

total_ingresos = (df[df["tipo"] == "Ingreso"].groupby("inicio_semana")["importe"].sum().reindex(flujo.columns, fill_value=0))
total_egresos = (df[df["tipo"] == "Egreso"].groupby("inicio_semana")["importe"].sum().reindex(flujo.columns, fill_value=0))

flujo_neto = total_ingresos - total_egresos

saldo_inicial = 1000000
saldo = saldo_inicial + flujo_neto.cumsum()

flujo.loc["TOTAL INGRESOS"] = total_ingresos
flujo.loc["TOTAL EGRESOS"] = -total_egresos
flujo.loc["FLUJO NETO"] = flujo_neto
flujo.loc["SALDO"] = saldo

nombres_semanas = (df[["inicio_semana", "semana"]].drop_duplicates().set_index("inicio_semana")["semana"].to_dict())

flujo.columns = [nombres_semanas[col] for col in flujo.columns]

flujo.index.name = "Concepto"

flujo = flujo.round(2)

archivo_salida = "Flujo de efectivo.xlsx"

with pd.ExcelWriter(archivo_salida, engine="openpyxl") as writer:
    flujo.to_excel(writer, sheet_name="Flujo")

    ws = writer.sheets["Flujo"]

    azul_encabezado = PatternFill(fill_type="solid", fgColor="1F4E78")
    azul_claro = PatternFill(fill_type="solid", fgColor="D9EAF7")
    azul_flujo = PatternFill(fill_type="solid", fgColor="9DC3E6")

    borde = Border(left=Side(style="thin"),right=Side(style="thin"),top=Side(style="thin"),bottom=Side(style="thin"))

    for row in ws.iter_rows():
        for cell in row:
            cell.border = borde
            cell.alignment = Alignment(horizontal="center", vertical="center")

    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = azul_encabezado

    for row in ws.iter_rows():
        if row[0].value in ["TOTAL INGRESOS", "TOTAL EGRESOS", "SALDO"]:
            for cell in row:
                cell.font = Font(bold=True)
                cell.fill = azul_claro

        if row[0].value == "FLUJO NETO":
            for cell in row:
                cell.font = Font(bold=True, size=13)
                cell.fill = azul_flujo

    for row in ws.iter_rows(min_row=2, min_col=2):
        for cell in row:
            cell.number_format = '#,##0.00'

    for col in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col[0].column)

        for cell in col:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[col_letter].width = max_length + 2

print("Archivo creado:", archivo_salida)

print(flujo)

Archivo creado: Flujo de efectivo.xlsx
                   03-08 noviembre  10-15 noviembre  17-22 noviembre  \
Concepto                                                               
1 KG FRESA                    0.00             0.00             0.00   
1 KG FRUTOS ROJOS             0.00             0.00             0.00   
2 KG MANGO                    0.00             0.00             0.00   
48 SNACK                      0.00             0.00             0.00   
AA                        -1800.00             0.00             0.00   
...                            ...              ...              ...   
VERO                          0.00             0.00             0.00   
TOTAL INGRESOS                0.00        303500.80        295000.00   
TOTAL EGRESOS           -943093.17       -311006.00       -292000.00   
FLUJO NETO              -943093.17         -7505.20          3000.00   
SALDO                     56906.83         49401.63         52401.63   

                   24-29